In [ ]:
import os
import itertools
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from scipy.signal import stft
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error
from sklearn.utils import shuffle
from tensorflow.keras.models import load_model, save_model

# ------------------------------------------------------------
# Global settings
# ------------------------------------------------------------
SEED = 3
np.random.seed(SEED)
tf.random.set_seed(SEED)

FS = 5000
DT = 1 / FS
N_PERSEG = int(0.025 * FS)
N_OVERLAP = int(0.021 * FS)
TIMEPOINTS = 138
FREQ_BINS = 63
BATCH_SIZE = 128
MAX_EPOCHS = 100

FREQS = [250, 500, 1000]
WEIGHT_UPDATES = [round(x, 3) for x in np.arange(0.005, 0.04, 0.005)]
PERT_LABELS = [f"{x * 100:.1f}%" for x in WEIGHT_UPDATES]

# Change this if your Drive path is different.
DRIVE_ROOT = Path('/content/drive/My Drive/S_D_ERP_RNN_final')
DATA_DIR = DRIVE_ROOT / 'data'
FINAL_DIR = DRIVE_ROOT / 'data_final'

print('TensorFlow version:', tf.__version__)
print('Drive root:', DRIVE_ROOT)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ------------------------------------------------------------
# Helper functions: loading and preprocessing empirical data
# ------------------------------------------------------------
def load_npy(path):
    return np.load(path, allow_pickle=True)


def to_ft(x):
    return np.asarray(x) * 1e15


def gaussian_noise(x, mu=0.0, std=0.6):
    return np.asarray(x) + np.random.normal(mu, std, size=np.asarray(x).shape)


def make_noisy_targets(base_signal, n_samples=1200, mu=0.0, std=0.6):
    base_signal = np.asarray(base_signal, dtype=np.float64)
    noisy = np.stack([gaussian_noise(base_signal, mu=mu, std=std) for _ in range(n_samples)], axis=0)
    return noisy


# Channel-averaged control data
standard_avg = load_npy(DRIVE_ROOT / 'avg_PRECED_S_array_meg_signif_ch_ctrl_signif_pat.npy')
deviant_avg = load_npy(DRIVE_ROOT / 'avg_D_array_meg_signif_ch_ctrl_signif_pat.npy')

if standard_avg.ndim > 1:
    standard_avg = np.mean(standard_avg, axis=0)
if deviant_avg.ndim > 1:
    deviant_avg = np.mean(deviant_avg, axis=0)

standard_avg_ft = to_ft(standard_avg)
deviant_avg_ft = to_ft(deviant_avg)

# If you want standardized versions instead, replace these two lines.
noisy_S = make_noisy_targets(standard_avg_ft, n_samples=1200, mu=0.0, std=0.6)
noisy_D = make_noisy_targets(deviant_avg_ft, n_samples=1200, mu=0.0, std=0.6)

print('Noisy target shapes:', noisy_S.shape, noisy_D.shape)


In [ ]:
# ------------------------------------------------------------
# Helper functions: stimulus generation and spectrograms
# ------------------------------------------------------------
def build_triplet_tone_sequence(freq_1, freq_2=None, fs=FS):
    """
    Create a three-tone sequence:
    standard: freq_1, freq_1, freq_1
    deviant:  freq_1, freq_1, freq_2
    """
    if freq_2 is None:
        freq_2 = freq_1

    tone_dur = 0.05
    gap_dur = 0.199

    n_tone = int(tone_dur * fs)
    n_gap = int(gap_dur * fs)

    t_tone = np.arange(n_tone) / fs
    tone1 = np.sin(2 * np.pi * freq_1 * t_tone)
    tone2 = np.sin(2 * np.pi * freq_1 * t_tone)
    tone3 = np.sin(2 * np.pi * freq_2 * t_tone)

    gap = np.zeros(n_gap)
    x = np.concatenate([tone1, gap, tone2, gap, tone3])
    return x


def compute_stft_abs(x, fs=FS, nperseg=N_PERSEG, noverlap=N_OVERLAP):
    f, t, zxx = stft(x, fs=fs, nperseg=nperseg, noverlap=noverlap)
    return f, t, np.abs(zxx)


def plot_spectrogram(zxx_abs, f, t, title=None, vmax=None):
    plt.figure(figsize=(6, 4))
    plt.pcolormesh(t * 1000, f, zxx_abs, vmin=0, vmax=vmax if vmax is not None else np.max(zxx_abs))
    plt.xlabel('Time (ms)', fontsize=14)
    plt.ylabel('Frequency (Hz)', fontsize=14)
    plt.ylim(0, 1200)
    plt.yticks(np.arange(0, 1250, 250))
    if title is not None:
        plt.title(title)
    plt.colorbar()
    plt.tight_layout()
    plt.show()


def tile_input(zxx_abs, n_repeats):
    # zxx_abs shape: (freq_bins, timepoints) -> transpose to (timepoints, freq_bins)
    spec = zxx_abs.T.astype(np.float64)
    return np.repeat(spec[None, :, :], repeats=n_repeats, axis=0)


def build_standard_inputs(freqs, repeats_per_freq=400, show_plots=True):
    all_specs = []
    for freq in freqs:
        x = build_triplet_tone_sequence(freq_1=freq, freq_2=None)
        f, t, zxx_abs = compute_stft_abs(x)
        if show_plots:
            plot_spectrogram(zxx_abs, f, t, title=f'Standard input: {freq} Hz')
        all_specs.append(tile_input(zxx_abs, repeats_per_freq))
    return np.concatenate(all_specs, axis=0)


def build_deviant_inputs(freqs, repeats_per_pair=200, show_plots=True):
    specs = []
    pairs = list(itertools.permutations(freqs, 2))
    for s_freq, d_freq in pairs:
        x = build_triplet_tone_sequence(freq_1=s_freq, freq_2=d_freq)
        f, t, zxx_abs = compute_stft_abs(x)
        if show_plots:
            plot_spectrogram(zxx_abs, f, t, title=f'Deviant input: {s_freq}-{d_freq} Hz')
        specs.append(tile_input(zxx_abs, repeats_per_pair))
    return np.concatenate(specs, axis=0), pairs


input_S = build_standard_inputs(FREQS, repeats_per_freq=400, show_plots=True)
input_D, deviant_pairs = build_deviant_inputs(FREQS, repeats_per_pair=200, show_plots=True)

print('input_S shape:', input_S.shape)
print('input_D shape:', input_D.shape)
print('deviant_pairs:', deviant_pairs)


In [ ]:
# ------------------------------------------------------------
# Build train / valid / test datasets
# ------------------------------------------------------------
def split_arrays(x, y, train_frac=0.7, val_frac=0.15):
    n = len(x)
    n_train = int(train_frac * n)
    n_val = int(val_frac * n)

    x_train = x[:n_train]
    y_train = y[:n_train]
    x_val = x[n_train:n_train + n_val]
    y_val = y[n_train:n_train + n_val]
    x_test = x[n_train + n_val:]
    y_test = y[n_train + n_val:]

    return x_train, y_train, x_val, y_val, x_test, y_test


def save_array(path, arr):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(path, np.asarray(arr, dtype=np.float64))


noisy_S = noisy_S.reshape(noisy_S.shape[0], noisy_S.shape[1], 1)
noisy_D = noisy_D.reshape(noisy_D.shape[0], noisy_D.shape[1], 1)

input_S, noisy_S = shuffle(input_S, noisy_S, random_state=0)
input_D, noisy_D = shuffle(input_D, noisy_D, random_state=1)

s_train_x, s_train_y, s_val_x, s_val_y, s_test_x, s_test_y = split_arrays(input_S, noisy_S)
d_train_x, d_train_y, d_val_x, d_val_y, d_test_x, d_test_y = split_arrays(input_D, noisy_D)

train_array = np.concatenate([s_train_x, d_train_x], axis=0)
labels_train_array = np.concatenate([s_train_y, d_train_y], axis=0)
valid_array = np.concatenate([s_val_x, d_val_x], axis=0)
labels_valid_array = np.concatenate([s_val_y, d_val_y], axis=0)
test_array = np.concatenate([s_test_x, d_test_x], axis=0)
labels_test_array = np.concatenate([s_test_y, d_test_y], axis=0)

labels_train_array = labels_train_array.reshape(labels_train_array.shape[0], labels_train_array.shape[1])
labels_valid_array = labels_valid_array.reshape(labels_valid_array.shape[0], labels_valid_array.shape[1])
labels_test_array = labels_test_array.reshape(labels_test_array.shape[0], labels_test_array.shape[1])

save_array(DATA_DIR / 'train_array.npy', train_array)
save_array(DATA_DIR / 'labels_train_array.npy', labels_train_array)
save_array(DATA_DIR / 'valid_array.npy', valid_array)
save_array(DATA_DIR / 'labels_valid_array.npy', labels_valid_array)
save_array(DATA_DIR / 'test_array.npy', test_array)
save_array(DATA_DIR / 'labels_test_array.npy', labels_test_array)

train_tf_dataset = tf.data.Dataset.from_tensor_slices((train_array, labels_train_array)).batch(BATCH_SIZE)
valid_tf_dataset = tf.data.Dataset.from_tensor_slices((valid_array, labels_valid_array)).batch(BATCH_SIZE)
test_tf_dataset = tf.data.Dataset.from_tensor_slices((test_array, labels_test_array)).batch(BATCH_SIZE)

example_batch = next(iter(train_tf_dataset))
print('Train array:', train_array.shape)
print('Labels:', labels_train_array.shape)
print('Example batch input shape:', example_batch[0].shape)
print('Example batch label shape:', example_batch[1].shape)


In [ ]:
# ------------------------------------------------------------
# Model definition, training, evaluation
# ------------------------------------------------------------
def build_simple_rnn_model(input_shape=(TIMEPOINTS, FREQ_BINS)):
    model = tf.keras.Sequential([
        tf.keras.layers.SimpleRNN(64, return_sequences=True, activation='relu', input_shape=input_shape),
        tf.keras.layers.SimpleRNN(64, return_sequences=True, activation='relu'),
        tf.keras.layers.SimpleRNN(64, return_sequences=True, activation='relu'),
        tf.keras.layers.SimpleRNN(64, return_sequences=True, activation='relu'),
        tf.keras.layers.SimpleRNN(1, return_sequences=True, activation='linear')
    ])
    return model


def compile_and_fit(model, train_data, valid_data, save_dir, model_prefix='simple_rnn_model_orig', patience=3):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_path = str(save_dir / 'model.{epoch:02d}-{val_loss:.4f}.h5')
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, mode='min'),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            save_weights_only=False,
            monitor='val_loss',
            mode='min',
            save_best_only=False,
            save_freq='epoch',
            verbose=1,
        ),
    ]

    optimizer = tf.keras.optimizers.Adam(learning_rate=0.00025)
    model.compile(optimizer=optimizer, loss='mean_squared_error', metrics=['mean_squared_error'])
    history = model.fit(train_data, epochs=MAX_EPOCHS, validation_data=valid_data, callbacks=callbacks)

    final_model_path = save_dir / f'{model_prefix}.h5'
    history_path = save_dir / f'{model_prefix}_history.npy'

    save_model(model, final_model_path)
    np.save(history_path, history.history)
    return history, final_model_path, history_path


simple_rnn_model = build_simple_rnn_model()
print('Input shape:', example_batch[0].shape)
print('Output shape:', simple_rnn_model(example_batch[0]).shape)

history, model_path, history_path = compile_and_fit(
    simple_rnn_model,
    train_data=train_tf_dataset,
    valid_data=valid_tf_dataset,
    save_dir=FINAL_DIR,
    model_prefix='simple_rnn_model_orig'
)

print('Saved model to:', model_path)
print('Saved history to:', history_path)

simple_rnn_model_orig = load_model(model_path)
score = simple_rnn_model_orig.evaluate(test_tf_dataset, verbose=2)
print('Test score:', score)


In [ ]:
# ------------------------------------------------------------
# Plot training history
# ------------------------------------------------------------
history_training_val = np.load(FINAL_DIR / 'simple_rnn_model_orig_history.npy', allow_pickle=True).item()
loss_train = history_training_val['loss']
loss_val = history_training_val['val_loss']
epochs = range(1, len(loss_train) + 1)

plt.figure(figsize=(6, 3))
plt.plot(epochs, loss_train, label='Training loss')
plt.plot(epochs, loss_val, label='Validation loss')
plt.xlabel('Epochs', fontsize=14)
plt.ylabel('Loss (MSE)', fontsize=14)
plt.legend()
plt.tight_layout()
plt.savefig(FINAL_DIR / 'training_valid_loss_orig.pdf')
plt.show()


In [ ]:
# ------------------------------------------------------------
# Output-layer predictions and hidden-layer activations
# ------------------------------------------------------------
TIME_MS = np.arange(0, 137 * 4 + 1, 4)


def build_activation_model(model, layer_index):
    return tf.keras.models.Model(inputs=model.input, outputs=model.layers[layer_index].output)


def predict_layer_output(model, x, layer_index):
    activation_model = build_activation_model(model, layer_index)
    return activation_model.predict(x.reshape(1, x.shape[0], x.shape[1]), verbose=0)


def average_selected_outputs(model, test_array, layer_index, indices):
    outputs = [predict_layer_output(model, test_array[idx], layer_index)[0, :, 0] for idx in indices]
    return np.mean(np.stack(outputs, axis=0), axis=0)


standard_indices = [1, 5, 10]
deviant_indices = [297, 267, 295]

avg_output_S_orig_model = average_selected_outputs(simple_rnn_model_orig, test_array, 4, standard_indices)
avg_output_D_orig_model = average_selected_outputs(simple_rnn_model_orig, test_array, 4, deviant_indices)

np.save(FINAL_DIR / 'avg_output_S_orig_model.npy', avg_output_S_orig_model)
np.save(FINAL_DIR / 'avg_output_D_orig_model.npy', avg_output_D_orig_model)

plt.figure(figsize=(6, 3))
plt.title('RNN Prediction')
plt.xlabel('Time (ms)')
plt.ylabel('Output Layer Activation')
plt.plot(TIME_MS, avg_output_S_orig_model, label='Standard', color='blue')
plt.plot(TIME_MS, avg_output_D_orig_model, label='Deviant', color='red')
plt.legend()
plt.tight_layout()
plt.savefig(FINAL_DIR / 'simple_rnn_model_orig_L5_prediction_S_D.pdf')
plt.show()


def plot_prediction_vs_target(avg_output_s, avg_output_d, labels_test_array, idx_s=1, idx_d=297):
    fig, axs = plt.subplots(2, figsize=(7, 5))
    axs[0].plot(TIME_MS, avg_output_s, label='RNN Prediction', color='cornflowerblue')
    axs[0].plot(TIME_MS, labels_test_array[idx_s], label='Model Target', color='darkblue', linestyle=':')
    axs[0].legend(fontsize=9)
    axs[0].set_ylim([-3, 3])
    axs[0].set_title('Standard Evoked Response')
    axs[0].set(xlabel='Time (ms)', ylabel='fT')

    axs[1].plot(TIME_MS, avg_output_d, label='RNN Prediction', color='turquoise')
    axs[1].plot(TIME_MS, labels_test_array[idx_d], label='Model Target', color='darkcyan', linestyle=':')
    axs[1].legend(fontsize=9)
    axs[1].set_ylim([-3, 3])
    axs[1].set_title('Deviant Evoked Response')
    axs[1].set(xlabel='Time (ms)', ylabel='fT')
    fig.tight_layout()
    return fig


plot_prediction_vs_target(avg_output_S_orig_model, avg_output_D_orig_model, labels_test_array)
plt.show()


def extract_hidden_activations(model, test_array, indices, n_hidden_layers=4):
    hidden_layers = np.zeros((n_hidden_layers, TIMEPOINTS, 64), dtype=np.float64)
    for layer_idx in range(n_hidden_layers):
        activation_model = build_activation_model(model, layer_idx)
        out = []
        for idx in indices:
            act = activation_model.predict(test_array[idx].reshape(1, TIMEPOINTS, FREQ_BINS), verbose=0)
            out.append(act[0])
        hidden_layers[layer_idx] = np.mean(np.stack(out, axis=0), axis=0)
    return hidden_layers


hidden_layers_activ_S = extract_hidden_activations(simple_rnn_model_orig, test_array, standard_indices)
hidden_layers_activ_D = extract_hidden_activations(simple_rnn_model_orig, test_array, deviant_indices)
np.save(FINAL_DIR / 'hidden_layers_activ_S_orig_model.npy', hidden_layers_activ_S)
np.save(FINAL_DIR / 'hidden_layers_activ_D_orig_model.npy', hidden_layers_activ_D)


def plot_hidden_heatmaps(hidden_layers, title):
    fig, axs = plt.subplots(4, figsize=(15, 7))
    for i in range(4):
        heat = axs[i].matshow(hidden_layers[i].T, cmap='viridis')
        axs[i].set_ylabel(f'Layer {i + 1} Units')
        axs[i].xaxis.set_ticks_position('bottom')
        axs[i].set_xticks([])
        axs[i].set_yticks([])
        plt.colorbar(heat)
    fig.suptitle(title, fontsize=15)
    fig.tight_layout()
    plt.show()


plot_hidden_heatmaps(hidden_layers_activ_S, 'Standard')
plot_hidden_heatmaps(hidden_layers_activ_D, 'Deviant')


def plot_hidden_unit_means(hidden_s, hidden_d):
    s_mean = hidden_s.mean(axis=2)
    d_mean = hidden_d.mean(axis=2)
    for l in range(4):
        plt.figure(figsize=(8, 3))
        plt.plot(TIME_MS, s_mean[l], label='Standard', color='blue')
        plt.plot(TIME_MS, d_mean[l], label='Deviant', color='red')
        plt.title(f'Hidden Layer {l + 1}')
        plt.legend(bbox_to_anchor=(1.1, 1), loc='upper right')
        plt.gca().spines['top'].set_visible(False)
        plt.gca().spines['right'].set_visible(False)
        plt.ylim(0, 0.1)
        plt.xlabel('Time')
        plt.ylabel('Activity')
        plt.tight_layout()
        plt.show()


plot_hidden_unit_means(hidden_layers_activ_S, hidden_layers_activ_D)


In [ ]:
# ------------------------------------------------------------
# Weight statistics
# ------------------------------------------------------------
def recurrent_weight_stats(model, n_hidden_layers=4):
    all_weights = []
    for l in range(n_hidden_layers):
        recurrent_w = model.layers[l].get_weights()[1].flatten()
        all_weights.append(recurrent_w)

        pos = recurrent_w[recurrent_w > 0]
        neg = recurrent_w[recurrent_w < 0]
        strength_pos = pos.sum() if len(pos) else 0.0
        strength_neg = neg.sum() if len(neg) else 0.0
        ratio_strength = strength_pos / np.abs(strength_neg) if strength_neg != 0 else np.nan
        print(f'Layer {l + 1} pos/neg strength ratio:', ratio_strength)

    all_weights = np.concatenate(all_weights)
    pos_total = all_weights[all_weights > 0]
    neg_total = all_weights[all_weights < 0]

    print('Pos weights total:', pos_total.shape[0])
    print('Neg weights total:', neg_total.shape[0])

    total_ratio = pos_total.shape[0] / neg_total.shape[0]
    print('Total count ratio pos/neg:', total_ratio)

    pos_strength = pos_total.sum()
    neg_strength = neg_total.sum()
    print('Total E/I strength ratio:', np.abs(pos_strength / neg_strength))

    avg_exc_syn = pos_strength / len(pos_total)
    avg_inh_syn = np.abs(neg_strength) / len(neg_total)
    print('Average synapse E/I ratio:', avg_exc_syn / avg_inh_syn)


recurrent_weight_stats(simple_rnn_model_orig)


### Generic perturbation framework

In [ ]:
# ------------------------------------------------------------
# Generic perturbation helpers
# ------------------------------------------------------------
def clone_model_from_disk(model_path):
    return load_model(model_path)


def apply_recurrent_perturbation(model, update_frac, mode='increase_negative', random_mask=None, n_hidden_layers=4):
    """
    mode options:
      - increase_negative
      - increase_positive
      - random_mixed
    """
    for l in range(n_hidden_layers):
        ff_w, rec_w, bias = model.layers[l].get_weights()
        rec_new = rec_w.copy()

        if mode == 'increase_negative':
            mask = rec_new < 0
            rec_new[mask] = rec_new[mask] + np.abs(update_frac * rec_new[mask])

        elif mode == 'increase_positive':
            mask = rec_new > 0
            rec_new[mask] = rec_new[mask] + np.abs(update_frac * rec_new[mask])

        elif mode == 'random_mixed':
            if random_mask is None:
                raise ValueError('random_mask must be provided for random_mixed mode.')
            rec_new = random_mask * rec_new
            neg_mask = rec_new < 0
            pos_mask = rec_new > 0
            rec_new[neg_mask] = rec_new[neg_mask] + np.abs(update_frac * rec_new[neg_mask])
            rec_new[pos_mask] = rec_new[pos_mask] + np.abs(update_frac * rec_new[pos_mask])
        else:
            raise ValueError(f'Unknown mode: {mode}')

        model.layers[l].set_weights([ff_w, rec_new, bias])
    return model


def evaluate_models_on_test(models, test_dataset):
    scores = []
    for m in models:
        score = m.evaluate(test_dataset, verbose=2)
        scores.append(score)
    return np.array(scores, dtype=object)


def save_models(models, out_dir, prefix, upd_list):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for model, upd in zip(models, upd_list):
        path = out_dir / f'{prefix}{upd}.h5'
        save_model(model, path)
        paths.append(path)
    return paths


def perturb_model_family(model_path, update_list, mode, out_prefix, random_mask=None):
    models = []
    for upd in update_list:
        m = clone_model_from_disk(model_path)
        m = apply_recurrent_perturbation(m, upd, mode=mode, random_mask=random_mask)
        models.append(m)

    scores = evaluate_models_on_test(models, test_tf_dataset)
    score_path = FINAL_DIR / f'scores_{out_prefix}.npy'
    np.save(score_path, scores)
    model_paths = save_models(models, FINAL_DIR, f'simple_rnn_model_{out_prefix}', update_list)
    return models, model_paths, scores, score_path


def plot_test_losses(scores, update_list, xlabel, title, out_path=None):
    losses = [float(s[0]) for s in scores]
    labels = [f'{u * 100:.1f}%' for u in update_list]
    plt.figure(figsize=(8, 4))
    plt.bar(labels, losses)
    plt.ylabel('Loss')
    plt.xlabel(xlabel)
    plt.title(title)
    plt.tight_layout()
    if out_path is not None:
        plt.savefig(out_path)
    plt.show()


def synapse_ei_ratios(model_paths, n_hidden_layers=4):
    for path in model_paths:
        model = load_model(path)
        print('\nModel:', path.name)
        for l in range(n_hidden_layers):
            rec_w = model.layers[l].get_weights()[1].flatten()
            pos = rec_w[rec_w > 0]
            neg = rec_w[rec_w < 0]
            avg_exc = pos.sum() / len(pos)
            avg_inh = np.abs(neg.sum()) / len(neg)
            print(f'Layer {l + 1} synapse E/I ratio:', np.abs(avg_exc / avg_inh))


def collect_avg_outputs_for_models(model_paths, test_array, standard_idx, deviant_idx, out_prefix):
    std_out = []
    dev_out = []
    for path in model_paths:
        model = load_model(path)
        std_out.append(average_selected_outputs(model, test_array, 4, standard_idx))
        dev_out.append(average_selected_outputs(model, test_array, 4, deviant_idx))

    std_out = np.stack(std_out, axis=0)
    dev_out = np.stack(dev_out, axis=0)
    np.save(FINAL_DIR / f'{out_prefix}_avg_output_S.npy', std_out)
    np.save(FINAL_DIR / f'{out_prefix}_avg_output_D.npy', dev_out)
    return std_out, dev_out


def plot_prediction_family(std_out, dev_out, avg_std_base, avg_dev_base, update_list, out_path=None):
    colors = ['rosybrown', 'indianred', 'coral', 'saddlebrown', 'cornflowerblue', 'slateblue', 'darkviolet', 'magenta']
    legend = [f'{u * 100:.1f}%' for u in update_list]

    plt.figure(figsize=(9.5, 6))
    plt.plot(TIME_MS, avg_std_base, color='black', linestyle='-', label='0%')
    plt.plot(TIME_MS, avg_dev_base, color='black', linestyle='--')
    for i in range(len(update_list)):
        plt.plot(TIME_MS, std_out[i], color=colors[i], linestyle='-', label=legend[i])
        plt.plot(TIME_MS, dev_out[i], color=colors[i], linestyle='--')
    plt.legend(loc='upper left', title='Perturbation level')
    plt.xlabel('Time (ms)', fontsize=17)
    plt.ylabel('RNN Prediction', fontsize=17)
    plt.tight_layout()
    if out_path is not None:
        plt.savefig(out_path)
    plt.show()


def peak_and_latency_metrics(avg_std_base, avg_dev_base, std_out, dev_out, out_prefix):
    pre_peak_s = np.max(avg_std_base)
    pre_peak_d = np.max(avg_dev_base)
    pre_mae = mean_absolute_error(avg_std_base, avg_dev_base)
    delta_t = 4

    ratio_s = []
    ratio_d = []
    mae_list = []
    lat_s = []
    lat_d = []

    for i in range(std_out.shape[0]):
        ratio_s.append(np.max(std_out[i]) / pre_peak_s)
        ratio_d.append(np.max(dev_out[i]) / pre_peak_d)
        mae_list.append(mean_absolute_error(std_out[i], dev_out[i]))
        lat_s.append((np.argmax(std_out[i]) - 1) * delta_t)
        lat_d.append((np.argmax(dev_out[i]) - 1) * delta_t)

    np.save(FINAL_DIR / f'{out_prefix}_peak_ampl_ratios_S.npy', np.array(ratio_s, dtype=np.float64))
    np.save(FINAL_DIR / f'{out_prefix}_peak_ampl_ratios_D.npy', np.array(ratio_d, dtype=np.float64))
    np.save(FINAL_DIR / f'{out_prefix}_MMN_MAE.npy', np.array(mae_list, dtype=np.float64))
    np.save(FINAL_DIR / f'{out_prefix}_peak_latency_S.npy', np.array(lat_s, dtype=np.float64))
    np.save(FINAL_DIR / f'{out_prefix}_peak_latency_D.npy', np.array(lat_d, dtype=np.float64))

    return {
        'pre_mae': pre_mae,
        'ratio_s': np.array(ratio_s, dtype=np.float64),
        'ratio_d': np.array(ratio_d, dtype=np.float64),
        'mmn_mae': np.array(mae_list, dtype=np.float64),
        'lat_s': np.array(lat_s, dtype=np.float64),
        'lat_d': np.array(lat_d, dtype=np.float64),
    }


def plot_ratio_summary(metrics, update_list, out_path=None):
    pert_vec = [0.0] + update_list
    x_labels = [f'{x * 100:.1f}%' for x in pert_vec]
    mmn_ratio = [1.0] + (metrics['mmn_mae'] / metrics['pre_mae']).tolist()
    ratio_s = [1.0] + metrics['ratio_s'].tolist()
    ratio_d = [1.0] + metrics['ratio_d'].tolist()

    plt.figure(figsize=(9.5, 6))
    plt.plot(pert_vec, ratio_s, linestyle='-', color='blue', label='Peak amplitude S', linewidth=3)
    plt.plot(pert_vec, ratio_d, linestyle='--', color='orange', label='Peak amplitude D', linewidth=3)
    plt.plot(pert_vec, mmn_ratio, linestyle='-', color='green', label='Mismatch negativity', linewidth=3)
    plt.xticks(pert_vec, x_labels, fontsize=14)
    plt.yticks(fontsize=14)
    plt.xlabel('Perturbation level', fontsize=16)
    plt.ylabel('Post- : Pre-perturbation ratio', fontsize=16)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.legend(fontsize=13)
    plt.tight_layout()
    if out_path is not None:
        plt.savefig(out_path)
    plt.show()


def pca_phase_diagrams(model_paths, test_array, layer_index=3, out_prefix='pca_activations', out_pdf=None):
    pca_list = []
    for path in model_paths:
        model = load_model(path)
        activation_model = build_activation_model(model, layer_index)
        acts = []
        for x in test_array:
            a = activation_model.predict(x.reshape(1, TIMEPOINTS, FREQ_BINS), verbose=0)
            acts.append(a)
        acts = np.array(acts)
        avg_acts = np.mean(acts, axis=0).reshape(-1, 64)
        pca = PCA(n_components=2)
        reduced = pca.fit_transform(avg_acts)
        pca_list.append(reduced)

    pca_list = np.array(pca_list)
    np.save(FINAL_DIR / f'{out_prefix}.npy', pca_list)

    fig, axs = plt.subplots(2, 4, figsize=(20, 8))
    fig.suptitle(f'Phase diagrams of hidden-layer activations: {out_prefix}', fontsize=20)
    for k in range(4):
        for f in range(1, TIMEPOINTS):
            axs[0, k].plot(pca_list[k][f - 1:f + 1, 0], pca_list[k][f - 1:f + 1, 1], color=plt.cm.viridis(f / TIMEPOINTS), linewidth=2)
            axs[1, k].plot(pca_list[k + 4][f - 1:f + 1, 0], pca_list[k + 4][f - 1:f + 1, 1], color=plt.cm.viridis(f / TIMEPOINTS), linewidth=2)
        axs[0, k].set_xlabel('PC1')
        axs[0, k].set_ylabel('PC2')
        axs[1, k].set_xlabel('PC1')
        axs[1, k].set_ylabel('PC2')
        axs[0, k].set_title(f'{WEIGHT_UPDATES[k] * 100:.1f}%')
        axs[1, k].set_title(f'{WEIGHT_UPDATES[k + 4] * 100:.1f}%')

    cax = plt.axes([1, 0.15, 0.02, 0.7])
    plt.colorbar(plt.cm.ScalarMappable(cmap=plt.cm.viridis), cax=cax, label='Timestep')
    fig.tight_layout()
    if out_pdf is not None:
        fig.savefig(out_pdf)
    plt.show()

    return pca_list


#### Experiment 1: increase negative recurrent weights

In [ ]:
orig_model_path = FINAL_DIR / 'simple_rnn_model_orig.h5'

neg_models, neg_model_paths, scores_neg, scores_neg_path = perturb_model_family(
    model_path=orig_model_path,
    update_list=WEIGHT_UPDATES,
    mode='increase_negative',
    out_prefix='perturb_neg_w'
)

plot_test_losses(
    scores_neg,
    WEIGHT_UPDATES,
    xlabel='Relative increase of recurrent negative weights',
    title='Loss across perturbation levels',
    out_path=FINAL_DIR / 'loss_neg_w_perturb.pdf'
)

synapse_ei_ratios(neg_model_paths)

neg_std_out, neg_dev_out = collect_avg_outputs_for_models(
    neg_model_paths,
    test_array,
    standard_indices,
    deviant_indices,
    out_prefix='neg_w_perturb'
)

plot_prediction_family(
    neg_std_out,
    neg_dev_out,
    avg_output_S_orig_model,
    avg_output_D_orig_model,
    WEIGHT_UPDATES,
    out_path=FINAL_DIR / 'neg_w_perturb_predictions_single_plot.pdf'
)

neg_metrics = peak_and_latency_metrics(
    avg_output_S_orig_model,
    avg_output_D_orig_model,
    neg_std_out,
    neg_dev_out,
    out_prefix='neg_w_perturb'
)

plot_ratio_summary(
    neg_metrics,
    WEIGHT_UPDATES,
    out_path=FINAL_DIR / 'neg_w_perturb_post_perturb_increase.pdf'
)

pca_phase_diagrams(
    neg_model_paths,
    test_array,
    layer_index=3,
    out_prefix='neg_w_perturb_PCA_L4_activations',
    out_pdf=FINAL_DIR / 'neg_w_perturb_PCA_L4_activations.pdf'
)


#### Experiment 2: increase positive recurrent weights

In [ ]:
pos_models, pos_model_paths, scores_pos, scores_pos_path = perturb_model_family(
    model_path=orig_model_path,
    update_list=WEIGHT_UPDATES,
    mode='increase_positive',
    out_prefix='perturb_pos_w'
)

plot_test_losses(
    scores_pos,
    WEIGHT_UPDATES,
    xlabel='Relative increase of recurrent positive weights',
    title='Loss across perturbation levels',
    out_path=FINAL_DIR / 'loss_pos_w_perturb.pdf'
)

synapse_ei_ratios(pos_model_paths)

pos_std_out, pos_dev_out = collect_avg_outputs_for_models(
    pos_model_paths,
    test_array,
    standard_indices,
    deviant_indices,
    out_prefix='control_1'
)

plot_prediction_family(
    pos_std_out,
    pos_dev_out,
    avg_output_S_orig_model,
    avg_output_D_orig_model,
    WEIGHT_UPDATES,
    out_path=FINAL_DIR / 'pos_w_perturb_predictions_single_plot.pdf'
)

pos_metrics = peak_and_latency_metrics(
    avg_output_S_orig_model,
    avg_output_D_orig_model,
    pos_std_out,
    pos_dev_out,
    out_prefix='control_1'
)

plot_ratio_summary(
    pos_metrics,
    WEIGHT_UPDATES,
    out_path=FINAL_DIR / 'control_1_post_perturb_increase.pdf'
)

pca_phase_diagrams(
    pos_model_paths,
    test_array,
    layer_index=3,
    out_prefix='pos_w_perturb_PCA_L4_activations',
    out_pdf=FINAL_DIR / 'pos_w_perturb_PCA_L4_activations.pdf'
)


#### Experiment 3: random mixed perturbation

In [ ]:
np.random.seed(1)
random_targets = np.random.randint(2, size=(64, 64))

mix_models, mix_model_paths, scores_mix, scores_mix_path = perturb_model_family(
    model_path=orig_model_path,
    update_list=WEIGHT_UPDATES,
    mode='random_mixed',
    out_prefix='control_2',
    random_mask=random_targets
)

plot_test_losses(
    scores_mix,
    WEIGHT_UPDATES,
    xlabel='Relative increase of a random subset of recurrent weights',
    title='Loss across perturbation levels',
    out_path=FINAL_DIR / 'loss_control_2.pdf'
)

synapse_ei_ratios(mix_model_paths)

mix_std_out, mix_dev_out = collect_avg_outputs_for_models(
    mix_model_paths,
    test_array,
    standard_indices,
    deviant_indices,
    out_prefix='control_2'
)

plot_prediction_family(
    mix_std_out,
    mix_dev_out,
    avg_output_S_orig_model,
    avg_output_D_orig_model,
    WEIGHT_UPDATES,
    out_path=FINAL_DIR / 'control_2_w_perturb_predictions_single_plot.pdf'
)

mix_metrics = peak_and_latency_metrics(
    avg_output_S_orig_model,
    avg_output_D_orig_model,
    mix_std_out,
    mix_dev_out,
    out_prefix='control_2_perturb'
)

plot_ratio_summary(
    mix_metrics,
    WEIGHT_UPDATES,
    out_path=FINAL_DIR / 'control_2_post_perturb_increase.pdf'
)

pca_phase_diagrams(
    mix_model_paths,
    test_array,
    layer_index=3,
    out_prefix='control_2_perturb_PCA_L4_activations',
    out_pdf=FINAL_DIR / 'control_2_perturb_PCA_L4_activations.pdf'
)
